# Los pingüinos de la tabla

**Nivel:** introductorio

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/narrative/blob/codex/corporate-data-narrative-lab/corporate-data-narrative-lab/outputs/notebooks/02-los-pinguinos-de-la-tabla.ipynb)

## Pregunta central

¿Qué puede afirmar el equipo sobre los pingüinos a partir de esta tabla, y qué no?

## Recreación narrativa

La jefa pidió una diapositiva titulada «Así son los pingüinos». Renata abrió 344 registros y preguntó: «¿Cuáles pingüinos?». Tomás señaló la pantalla: «Los de la tabla». Celia miró las ocho columnas y resumió el problema: «La tabla tiene 344 filas; la Antártida, por ahora, se niega a caber en Excel». El equipo decide reconstruir la afirmación desde la unidad más pequeña hasta la decisión de generalizar.

<img src="https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png" alt="Ilustración de pingüinos Adelie, Chinstrap y Gentoo del archipiélago Palmer." style="max-width:100%;height:auto">

<small>Imagen: <a href="https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/vignettes/art.Rmd">Artwork by @allison_horst</a>, CC0 1.0 Universal.</small>

*La escena es una recreación; las conclusiones provienen del dataset citado.*

## Fuente real

**Palmer Archipelago (Antarctica) penguin data — penguins.csv**, Palmer Station LTER; paquete palmerpenguins de Horst, Hill y Gorman. [Página de origen](https://allisonhorst.github.io/palmerpenguins/) · [datos](https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv) · licencia: CC0 1.0 Universal.  
Consultado: 2026-07-18 · 344 filas · columnas usadas: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`, `year`.

In [1]:
# @title Preparar los datos { display-mode: "form" }
especie = "Todas" # @param ["Todas", "Adelie", "Chinstrap", "Gentoo"]
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
DATA_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
df_completo = pd.read_csv(DATA_URL)
df = df_completo if especie == "Todas" else df_completo.query("species == @especie")
df = df.reset_index(drop=True)
display(Markdown(f"**Filtro global:** {especie} · **{len(df)} observaciones**"))
df.head()

**Filtro global:** Todas · **344 observaciones**

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## 1. Observación

Renata elige la primera fila. No es «el pingüino promedio»: es un registro individual con especie, isla, medidas, sexo y año.

**Pregunta:** ¿Qué representa una fila?

**Conexión:** Punto de partida: identificar qué representa una fila antes de resumirla.

In [2]:
observacion = df.iloc[[0]]
observacion

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007


In [3]:
# @title Explorar Observación { display-mode: "form" }
vista = df.dropna(subset=["flipper_length_mm", "body_mass_g"])
punto = vista.iloc[0]
fig = px.scatter(vista, x="flipper_length_mm", y="body_mass_g", hover_data=["species", "island", "sex", "year"], title="Cada punto es una observación registrada")
fig.add_scatter(x=[punto["flipper_length_mm"]], y=[punto["body_mass_g"]], mode="markers", name="Una observación", marker={"size": 15, "color": "#F28E2B", "symbol": "diamond"})
fig.update_layout(updatemenus=[{"buttons": [{"label": "Todas", "method": "update", "args": [{"visible": [True, False]}]}, {"label": "Resaltar una", "method": "update", "args": [{"visible": [True, True]}]}]}], xaxis_title="aleta (mm)", yaxis_title="masa (g)", template="plotly_white")
fig.show()

display(Markdown("**Lo que muestra:** Un punto no es una especie ni un promedio: es una unidad registrada. Hover muestra los valores que viajan juntos en esa fila; el botón separa una observación del conjunto."))

**Lo que muestra:** Un punto no es una especie ni un promedio: es una unidad registrada. Hover muestra los valores que viajan juntos en esa fila; el botón separa una observación del conjunto.

## 2. Variable

Tomás descubre que «columna» describe la posición; «variable» describe qué se midió. Y algunas mediciones llegan vacías, porque hasta el dato real tiene días administrativos.

**Pregunta:** ¿Qué cambia de una columna a otra?

**Conexión:** La Observación contiene varios atributos; ahora identificamos las variables que les dan nombre y tipo.

In [4]:
variables = pd.DataFrame({
    "variable": observacion.columns,
    "tipo": ["numérica" if pd.api.types.is_numeric_dtype(df[c]) else "categórica" for c in observacion.columns],
    "faltantes": [df[c].isna().sum() for c in observacion.columns]
})
variables

,variable,tipo,faltantes
0,species,categórica,0
1,island,categórica,0
2,bill_length_mm,numérica,2
3,bill_depth_mm,numérica,2
4,flipper_length_mm,numérica,2
5,body_mass_g,numérica,2
6,sex,categórica,11
7,year,numérica,0


In [5]:
# @title Explorar Variable { display-mode: "form" }
faltan = df.isna().sum()
completos = len(df) - faltan
fig = go.Figure([go.Bar(x=df.columns, y=completos, name="Completos"), go.Bar(x=df.columns, y=faltan, name="Faltantes")])
fig.update_layout(barmode="stack", title="Calidad por variable", template="plotly_white", updatemenus=[{"buttons": [{"label": "Conteos", "method": "update", "args": [{"y": [completos.values, faltan.values]}, {"yaxis": {"title": "celdas"}}]}, {"label": "Porcentaje", "method": "update", "args": [{"y": [100 * completos.values / len(df), 100 * faltan.values / len(df)]}, {"yaxis": {"title": "% de filas"}}]}]}])
fig.show()

display(Markdown("**Lo que muestra:** Las ocho variables no son intercambiables: unas son categóricas y otras numéricas. El control permite ver faltantes como conteos o porcentajes sin cambiar la tabla de origen."))

**Lo que muestra:** Las ocho variables no son intercambiables: unas son categóricas y otras numéricas. El control permite ver faltantes como conteos o porcentajes sin cambiar la tabla de origen.

## 3. Tabla

Celia deja de ver «muchos números» y ve una cuadrícula: filas por observaciones, columnas por variables y algunos huecos que no deben maquillarse con entusiasmo.

**Pregunta:** ¿Cómo se organiza el conjunto completo?

**Conexión:** Al repetir la misma Variable para muchas observaciones obtenemos una estructura rectangular: la tabla.

In [6]:
resumen_tabla = pd.Series({
    "filas": len(df), "variables": len(variables),
    "celdas": df.size, "faltantes": int(df.isna().sum().sum())
})
resumen_tabla

filas         344
variables       8
celdas       2752
faltantes      19
dtype: int64

In [7]:
# @title Explorar Tabla { display-mode: "form" }
presencia = df.head(40).notna().astype(int).T
completos = go.Heatmap(z=presencia.values, x=presencia.columns, y=presencia.index, colorscale=[[0, "#F28E2B"], [1, "#4E79A7"]], name="Completitud", hovertemplate="variable=%{y}<br>fila=%{x}<br>presente=%{z}<extra></extra>")
huecos = go.Heatmap(z=1 - presencia.values, x=presencia.columns, y=presencia.index, colorscale=[[0, "#E5E7EB"], [1, "#E15759"]], name="Faltantes", visible=False, hovertemplate="variable=%{y}<br>fila=%{x}<br>faltante=%{z}<extra></extra>")
fig = go.Figure([completos, huecos])
fig.update_layout(title="Primeras 40 filas de la tabla", template="plotly_white", updatemenus=[{"buttons": [{"label": "Completitud", "method": "update", "args": [{"visible": [True, False]}]}, {"label": "Sólo huecos", "method": "update", "args": [{"visible": [False, True]}]}]}])
fig.show()

display(Markdown("**Lo que muestra:** La tabla completa tiene filas × variables celdas; el mapa enseña las primeras 40. Los huecos pertenecen a variables concretas y deben conservarse como información de calidad."))

**Lo que muestra:** La tabla completa tiene filas × variables celdas; el mapa enseña las primeras 40. Los huecos pertenecen a variables concretas y deben conservarse como información de calidad.

## 4. Población

La jefa propone «todos los pingüinos». Renata reduce la ambición: primero definamos especies, colonias, lugar y periodo. El universo no acepta comodines en la letra pequeña.

**Pregunta:** ¿Sobre qué conjunto querríamos concluir?

**Conexión:** La Tabla delimita lo observado, pero no define automáticamente a quiénes queremos representar: esa es la población.

In [8]:
poblacion_objetivo = pd.Series({
    "filas_observadas": resumen_tabla["filas"],
    "especies_observadas": df["species"].nunique(),
    "islas_observadas": df["island"].nunique(),
    "periodo_observado": f'{df["year"].min()}–{df["year"].max()}'
})
poblacion_objetivo

filas_observadas             344
especies_observadas            3
islas_observadas               3
periodo_observado      2007–2009
dtype: object

In [9]:
# @title Explorar Población { display-mode: "form" }
cobertura = df.groupby(["species", "island", "year"], observed=True, as_index=False).size()
detalle = px.sunburst(cobertura, path=["species", "island", "year"], values="size").data[0]
especies = px.sunburst(cobertura, path=["species"], values="size").data[0]
especies.visible = False
fig = go.Figure([detalle, especies])
fig.update_layout(title="Cobertura observada, no tamaño poblacional", template="plotly_white", updatemenus=[{"buttons": [{"label": "Especie → isla → año", "method": "update", "args": [{"visible": [True, False]}]}, {"label": "Sólo especies", "method": "update", "args": [{"visible": [False, True]}]}]}])
fig.show()

display(Markdown("**Lo que muestra:** El gráfico permite recorrer la cobertura de los registros. No cuenta la población ni a quienes quedaron fuera: el CSV no contiene esa información ni prueba representatividad."))

**Lo que muestra:** El gráfico permite recorrer la cobertura de los registros. No cuenta la población ni a quienes quedaron fuera: el CSV no contiene esa información ni prueba representatividad.

## 5. Muestra

Tomás cuenta 152 Adelie, 124 Gentoo y 68 Chinstrap. «Entonces ya está». Celia corrige: «Ya está el conteo; la representatividad todavía no ha enviado su currículum».

**Pregunta:** ¿Cómo está compuesta la muestra observada?

**Conexión:** Frente a la Población objetivo, las filas disponibles constituyen la muestra que realmente podemos analizar.

In [10]:
muestra = df.groupby("species").size().rename("n").to_frame()
muestra["% de la tabla"] = (
    100 * muestra["n"] / poblacion_objetivo["filas_observadas"]
).round(1)
muestra

,n,% de la tabla
species,,
Adelie,152,44.2
Chinstrap,68,19.8
Gentoo,124,36.0


In [11]:
# @title Explorar Muestra { display-mode: "form" }
composicion = df.groupby("species", observed=True, as_index=False).size().rename(columns={"size": "n"})
composicion["porcentaje"] = 100 * composicion["n"] / len(df)
fig = go.Figure(go.Bar(x=composicion["species"], y=composicion["n"], text=composicion["n"], customdata=composicion[["porcentaje"]]))
fig.update_traces(hovertemplate="%{x}<br>n=%{y}<br>%=%{customdata[0]:.1f}<extra></extra>")
fig.update_layout(title="Composición de la muestra", yaxis_title="observaciones", template="plotly_white", updatemenus=[{"buttons": [{"label": "Conteos", "method": "update", "args": [{"y": [composicion["n"]], "text": [composicion["n"]]}, {"yaxis": {"title": "observaciones"}}]}, {"label": "Porcentaje", "method": "update", "args": [{"y": [composicion["porcentaje"]], "text": [composicion["porcentaje"].round(1)]}, {"yaxis": {"title": "% de la tabla"}}]}]}])
fig.show()

display(Markdown("**Lo que muestra:** Con el filtro «Todas», la muestra suma 344 registros y su composición es desigual. Son porcentajes de la tabla, no fracciones de una población cuyo tamaño desconocemos."))

**Lo que muestra:** Con el filtro «Todas», la muestra suma 344 registros y su composición es desigual. Son porcentajes de la tabla, no fracciones de una población cuyo tamaño desconocemos.

## Cómo se conecta todo

Una observación reúne valores de varias variables. Muchas observaciones alineadas bajo las mismas variables forman una tabla. Esa tabla describe la muestra observada; la población es el conjunto sobre el que querríamos concluir. Aquí vemos cobertura por especie, isla y año, pero no el tamaño ni a los miembros no observados de esa población.

## Decisión

Cambiar el titular por «Patrones en 344 pingüinos observados del archipiélago Palmer entre 2007 y 2009» y pedir el diseño de muestreo antes de generalizar. Los pingüinos ausentes de la tabla tuvieron la cortesía estadística de no confirmar nada.

**Regla:** Una tabla describe su muestra; solo un diseño de muestreo defendible permite extender conclusiones a la población.